# 0. 환경설정

In [1]:
import os
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.tree import plot_tree
from sklearn.model_selection import GridSearchCV, StratifiedShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import AdaBoostClassifier
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression

from sklearn.svm import SVC

In [2]:
os.chdir('/content/drive/MyDrive/Colab Notebooks/06 머신러닝/Day_04_file')
os.getcwd()

'/content/drive/MyDrive/Colab Notebooks/06 머신러닝/Day_04_file'

# 1. 레드와인 등급 실습

In [3]:
# 실습 1-0 레드와인 데이터 읽기

df = pd.read_csv('redwine_quality.csv')
df.head()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
0,11.6,0.580,0.66,2.20,0.074,10.0,47.0,1.00080,3.25,0.57,9.0,3
1,10.4,0.610,0.49,2.10,0.200,5.0,16.0,0.99940,3.16,0.63,8.4,3
2,7.4,1.185,0.00,4.25,0.097,5.0,14.0,0.99660,3.63,0.54,10.7,3
3,10.4,0.440,0.42,1.50,0.145,34.0,48.0,0.99832,3.38,0.86,9.9,3
4,8.3,1.020,0.02,3.40,0.084,6.0,11.0,0.99892,3.48,0.49,11.0,3


In [4]:
# 실습 1-1 레드와인 데이터 전처리

df.replace({'quality':{3:5,4:5,5:5,6:6,7:7,8:7}}, inplace=True)
df.sample(10)

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
749,6.9,0.685,0.00,2.5,0.105,22.0,37.0,0.99660,3.46,0.57,10.6,6
757,7.3,0.390,0.31,2.4,0.074,9.0,46.0,0.99620,3.41,0.54,9.4,6
310,7.3,0.365,0.49,2.5,0.088,39.0,106.0,0.99660,3.36,0.78,11.0,5
591,7.5,0.580,0.14,2.2,0.077,27.0,60.0,0.99630,3.28,0.59,9.8,5
740,6.2,0.460,0.29,2.1,0.074,32.0,98.0,0.99578,3.33,0.62,9.8,5
278,12.3,0.390,0.63,2.3,0.091,6.0,18.0,1.00040,3.16,0.49,9.5,5
127,7.0,0.500,0.25,2.0,0.070,3.0,22.0,0.99630,3.25,0.63,9.2,5
11,5.7,1.130,0.09,1.5,0.172,7.0,19.0,0.99400,3.50,0.48,9.8,5
218,7.7,0.410,0.76,1.8,0.611,8.0,45.0,0.99680,3.06,1.26,9.4,5
1118,7.9,0.310,0.32,1.9,0.066,14.0,36.0,0.99364,3.41,0.56,12.6,6


In [5]:
# 실습 1-2 data, target 구분

y_data = df['quality']
x_data = df.drop('quality', axis = 1)
print(x_data.shape, y_data.shape)

(1599, 11) (1599,)


In [6]:
# 실습 1-3 3일차에서 선정된 모델
'''
RandomForestClassifier()
criterion' :'entropy'
n_estimators' : 200

'''

"\nRandomForestClassifier()\ncriterion' :'entropy'\nn_estimators' : 200\n\n"

In [7]:
# 실습 1-4 학습 시키기
wine_rf = RandomForestClassifier(criterion='entropy', n_estimators=200)

wine_rf.fit(x_data, y_data)

RandomForestClassifier(criterion='entropy', n_estimators=200)

In [8]:
df.columns

Index(['fixed acidity', 'volatile acidity', 'citric acid', 'residual sugar',
       'chlorides', 'free sulfur dioxide', 'total sulfur dioxide', 'density',
       'pH', 'sulphates', 'alcohol', 'quality'],
      dtype='object')

In [9]:
# 실습 1-5 학습된 모델로 함수 적

def wine_quality(data) :
  pred = wine_rf.predict(data)
  if pred <= 5:
    ret = '5등급 이하입니다.'
  elif pred >= 7:
    ret = '7등급 이상입니다.'
  else :
    ret = '6등급입니다.'
  return ret

wine_quality(x_data[:1])

'5등급 이하입니다.'

In [10]:
# 실습 1-6 신뢰도 향상 방법

df[['quality']].value_counts()

,count
quality,
5,744
6,638
7,217


In [11]:
# 실습 1-6 신뢰도 향상 방법

# confusion_matrix를 통해서 잘 맞추는 것만 찾아도 됨

pred = wine_rf.predict(x_data)

print(confusion_matrix(y_data, pred))

[[744   0   0]
 [  0 638   0]
 [  0   0 217]]


# 2. Adaptive Boosting

In [12]:
# 실습 2-0 코드 뜯어보기

'''
AdaBoostClassifier(

estimator=None,         # 사용할 모델[기본값 : ecisionTreeClassifier(max_depth=1)] 조정 소요 별로 없음
*,
n_estimators=50,        # 학습기 트리의 개수 -> 가중치

learning_rate=1.0,      # 가중치의 강도 : 1.0이면 적은 수의 트리로 수렴, 작아질수록 모델이 정교해지고 많은 가중치가 필요함

algorithm='deprecated', # 신경쓸 필요 없음

random_state=None       # 랜덤 시드값 조정

)
'''

"\nAdaBoostClassifier(\n\nestimator=None,         # 사용할 모델[기본값 : ecisionTreeClassifier(max_depth=1)] 조정 소요 별로 없음\n*, \nn_estimators=50,        # 학습기 트리의 개수 -> 가중치\n\nlearning_rate=1.0,      # 가중치의 강도 : 1.0이면 적은 수의 트리로 수렴, 작아질수록 모델이 정교해지고 많은 가중치가 필요함\n\nalgorithm='deprecated', # 신경쓸 필요 없음\n\nrandom_state=None       # 랜덤 시드값 조정\n\n)\n"

In [13]:
# 실습 2-1 데이터 구분 및 모델 적용

x_train, x_test, y_train, y_test = train_test_split(x_data,
                                                    y_data,
                                                    stratify=y_data)

wine_abd = AdaBoostClassifier()
wine_abd.fit(x_train, y_train)
pred = wine_abd.predict(x_test)

print(accuracy_score(y_test, pred))
print(confusion_matrix(y_test, pred))

0.6175
[[139  44   3]
 [ 57  84  19]
 [  5  25  24]]


In [14]:
# 실습 2-2 모델의 파라미터 조정

for est in range(50,501,50) :

  wine_abd = AdaBoostClassifier(n_estimators=est)
  wine_abd.fit(x_train, y_train)
  pred = wine_abd.predict(x_test)

  print(accuracy_score(y_test, pred))
  print(confusion_matrix(y_test, pred))

0.6175
[[139  44   3]
 [ 57  84  19]
 [  5  25  24]]
0.615
[[140  42   4]
 [ 58  83  19]
 [  6  25  23]]
0.6175
[[133  50   3]
 [ 52  90  18]
 [  6  24  24]]
0.62
[[138  44   4]
 [ 55  84  21]
 [  7  21  26]]
0.6275
[[138  44   4]
 [ 54  86  20]
 [  7  20  27]]
0.62
[[136  46   4]
 [ 53  87  20]
 [  6  23  25]]
0.6275
[[135  47   4]
 [ 50  92  18]
 [  6  24  24]]
0.6325
[[140  42   4]
 [ 52  87  21]
 [  6  22  26]]
0.62
[[139  42   5]
 [ 54  84  22]
 [  6  23  25]]
0.635
[[138  43   5]
 [ 51  91  18]
 [  5  24  25]]


# 3. 소프트맥스 회귀(다항 로지스틱 회귀)

In [15]:
# 실습 3-0 코드 뜯어보기

'''
LogisticRegression(

* penalty='l2',          #'l2'는 가중치들의 제곱합을 최소화하는 릿지 방식
* C=1.0,                 # 규제강도 : 작을 수록 단순해짐
* solver='lbfgs',        # 최적의 기울기를 찾는 알고리즘
* max_iter=100,          # 학습을 시도하는 최대 횟수

tol=0.0001,              # 모델의 성능 개선 폭이 이 값(0.0001)보다 작아지면 멈추는 기준점
fit_intercept=True,      # 절편(y절편)을 포함할지 결정
intercept_scaling=1,     # 데이터셋에 절편(Intercept)을 추가할 때, 절편 값을 어느 정도로 스케일링(조절)할 것인가'를 결정
class_weight=None,       # 데이터가 불균형할 때(예: 정상이 99개, 불량이 1개) 특정 클래스에 가중치를 주는 설정
random_state=None,       # 랜덤 시드값
verbose=0,               # 진행로그
warm_start=False,        # 이전에 학습했던 모델의 상태를 이어받아 계속 학습할지 결정
n_jobs=None,             # 병렬 처리를 할 CPU 개수
l1_ratio=None            # 규제 방식 중 'Elastic Net(L1과 L2의 결합)'을 사용할 때, L1과 L2 규제를 각각 몇 퍼센트씩 섞을지 정하는 비율
)
'''

"\nLogisticRegression(\n\n* penalty='l2',          #'l2'는 가중치들의 제곱합을 최소화하는 릿지 방식\n* C=1.0,                 # 규제강도 : 작을 수록 단순해짐\n* solver='lbfgs',        # 최적의 기울기를 찾는 알고리즘\n* max_iter=100,          # 학습을 시도하는 최대 횟수\n\ntol=0.0001,              # 모델의 성능 개선 폭이 이 값(0.0001)보다 작아지면 멈추는 기준점\nfit_intercept=True,      # 절편(y절편)을 포함할지 결정\nintercept_scaling=1,     # 데이터셋에 절편(Intercept)을 추가할 때, 절편 값을 어느 정도로 스케일링(조절)할 것인가'를 결정\nclass_weight=None,       # 데이터가 불균형할 때(예: 정상이 99개, 불량이 1개) 특정 클래스에 가중치를 주는 설정\nrandom_state=None,       # 랜덤 시드값\nverbose=0,               # 진행로그 \nwarm_start=False,        # 이전에 학습했던 모델의 상태를 이어받아 계속 학습할지 결정\nn_jobs=None,             # 병렬 처리를 할 CPU 개수\nl1_ratio=None            # 규제 방식 중 'Elastic Net(L1과 L2의 결합)'을 사용할 때, L1과 L2 규제를 각각 몇 퍼센트씩 섞을지 정하는 비율\n)\n"

In [16]:
# 실습 3-1

np.unique(y_data, return_counts=True)

(array([5, 6, 7]), array([744, 638, 217]))

In [17]:
# 실습 3-2 로지스틱 적용(Breast Cancer)

bc = load_breast_cancer()

x_train, x_test, y_train, y_test = train_test_split(bc.data,
                                                    bc.target,
                                                    stratify=bc.target)

# 표준화 전처리

scaler = StandardScaler()
x_train_scaled = scaler.fit_transform(x_train)
x_test_scaled = scaler.transform(x_test)

# 학습 및 예측

lr = LogisticRegression()
lr.fit(x_train_scaled, y_train)
pred = lr.predict(x_test_scaled)

print(accuracy_score(y_test, pred))

0.965034965034965


In [18]:
# 실습 3-3 로지스틱 적용(Iris)

iris = load_iris()

x_train, x_test, y_train, y_test = train_test_split(iris.data,
                                                    iris.target,
                                                    stratify=iris.target)

# 표준화 전처리

scaler = StandardScaler()
x_train_scaled = scaler.fit_transform(x_train)
x_test_scaled = scaler.transform(x_test)

# 학습 및 예측

lr = LogisticRegression()
lr.fit(x_train_scaled, y_train)
pred = lr.predict(x_test_scaled)

print(accuracy_score(y_test, pred))

0.9473684210526315


# 4. 서포트 벡터 머신

In [25]:
# 실습 4-0 코드 뜯어보기
'''
SVC(

* C=1.0,                         # 에러의 허용치
* kernel='rbf',                  # 커널함수
* degree=3,                      # 3차식으로 진행
* gamma='scale',                 # 클수록 데이터포인트의 영향력이 작아짐
coef0=0.0,                       # 곡선 경계가 너무 복잡하게 꼬일 때, 이 값을 조정해서 경계를 조금 더 안정적으로 잡아주는 역할
shrinking=True,                  # '정답 후보가 아닌 것 같은 데이터'는 빠르게 계산에서 제외해서 학습 속도를 훨씬 빠르게
probability=False,               # 데이터가 A일 구체적인 확률이 필요하다면 True로 설정
tol=0.001,                       # 오차 수용 기준치
cache_size=200,                  # 계산할 때 행렬을 메모리용량
class_weight=None,               # 적은 데이터에 가중치를 줘서 공평하게 학습하도록 강제 함
verbose=False,
max_iter=-1,                     # 횟수를 지정해서 강제로 끊어
decision_function_shape='ovr',   # 데이터를 둘 중 하나로 분류하는 걸 기본으로 합니다. 3개 이상의 등급을 나눌 때 각각을 하나씩 떼어내어 분류함
break_ties=False,                # 등급 분류를 했는데 두가지 종류의 확률이 완전히 똑같을 때 하나로 강제함
random_state=None                # 시드값
)
'''

"\nSVC(\n\n* C=1.0,                         # 에러의 허용치\n* kernel='rbf',                  # 커널함수\n* degree=3,                      # 3차식으로 진행\n* gamma='scale',                 # 클수록 데이터포인트의 영향력이 작아짐\ncoef0=0.0,                       # \nshrinking=True,                  # '정답 후보가 아닌 것 같은 데이터'는 빠르게 계산에서 제외해서 학습 속도를 훨씬 빠르게 \nprobability=False,               # \ntol=0.001,                       # 오차 수용 기준치\ncache_size=200,                  # 계산할 때 행렬을 메모리용량\nclass_weight=None,               # \nverbose=False, \nmax_iter=-1,                     \ndecision_function_shape='ovr', \nbreak_ties=False, \nrandom_state=None              # 시드값\n)\n"

In [20]:
## 실습 4-1 SVM 아이리스 적용

iris_svc = SVC()

iris_svc.fit(x_train_scaled, y_train)
pred = iris_svc.predict(x_test_scaled)

print(accuracy_score(y_test, pred))
print(confusion_matrix(y_test, pred))

0.9473684210526315
[[13  0  0]
 [ 0 11  2]
 [ 0  0 12]]


In [21]:
# 실습 4-2 파이프라인을 통한 도출

# kernel = 'linear', 'poly', 'rbf'
# gamma = [1,10,50,100]
# c= [0.01, 1, 100]


In [22]:
# 실습 4-3 파이프라인을 통한 도출(와인 등급)

y_data = df['quality']
x_data = df.drop('quality', axis = 1)

# Train, Test 데이터 분할
x_train, x_test, y_train, y_test = train_test_split(x_data,
                                                    y_data,
                                                    random_state=42,
                                                    stratify=y_data       # stratify=y : 원본 데이터의 등급(quality) 비율을 그대로 유지하면서 분할합니다.
                                                    )
print(x_train.shape, y_train.shape)

(1199, 11) (1199,)


In [24]:
# 실습 4-3 파이프라인을 통한 도출(와인 등급)

'''
폴리 감마 10 이상은 어마어마하다..홀리몰리...
'''

# pipe = Pipeline([
#     ('preprocess', StandardScaler()),
#     ('classifier', AdaBoostClassifier())
# ])
# param_grid = [
#     # Adaptive Boosting
#     {'preprocess': [StandardScaler()],
#      'classifier': [AdaBoostClassifier(random_state=42)],
#      'classifier__n_estimators': [50, 100, 200],
#      'classifier__learning_rate': [0.01, 0.1, 1.0]},
#     # LogisticRegression
#     {'preprocess': [StandardScaler()],
#      'classifier': [LogisticRegression(random_state=42)],
#      'classifier__C': [0.01, 0.1, 1, 10, 100],
#      'classifier__max_iter': [100, 500, 1000],
#      'classifier__penalty': ['l2']},
#     # SVC()
#     {'preprocess': [StandardScaler()],
#      'classifier': [SVC(random_state=42)],
#      'classifier__kernel': ['linear'],
#      'classifier__C': [0.01, 1, 10, 100]},

#     {'preprocess': [StandardScaler()],
#      'classifier': [SVC(random_state=42)],
#      'classifier__kernel': ['rbf'],
#      'classifier__gamma': [0.1, 1, 10, 50, 100],
#      'classifier__C': [0.01, 1, 10, 100]},

#     {'preprocess': [StandardScaler()],
#     'classifier': [SVC(random_state=42)],
#     'classifier__kernel': ['poly'],
#     'classifier__gamma': [0.1, 1, 10, 20],
#     'classifier__C': [0.01, 0.1, 1, 10, 100]},

#     # Random Forest
#     {'preprocess': [None],
#      'classifier': [RandomForestClassifier(random_state=42)],
#      'classifier__n_estimators': range(50, 201, 50),
#      "classifier__criterion":['gini', 'entropy']}
# ]


# # GridSearchCV 실행
# grid = GridSearchCV(pipe,
#                     param_grid,
#                     cv=5,
#                     return_train_score=True,
#                     verbose=3)

# # 학습 시작
# grid.fit(x_train, y_train)

# print(grid.best_params_, grid.best_score_)





